# Pipeline 1: Zero-Shot Classification

Zero-shot classification lets a model classify text into categories it was **never explicitly trained on**.
Instead of learning 'this is positive / negative', the model understands the *meaning* of the label names.

Under the hood it uses a Natural Language Inference (NLI) model:
it checks whether the sentence *entails* each candidate label.

In [1]:
from transformers import pipeline

/home/v/Desktop/Projects/001-transformers-course/.env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load the environment variables

Load dotenv to be able to access the HuggingFace API token.

In [2]:
from dotenv import load_dotenv
import os

load_dotenv(".secrets")
hf_token = os.getenv("HF_TOKEN")

## Load the pipeline

No model specified → HuggingFace picks a sensible default for the task.
First run downloads the model weights (~1.5 GB); subsequent runs use the local cache.

In [3]:
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", token=hf_token)

Loading weights: 100%|██████████| 515/515 [00:00<00:00, 10821.16it/s]


## Basic example — single label wins

In [4]:
text = "The new battery lasts 48 hours on a single charge."
candidate_labels = ["technology", "sports", "cooking"]

result = classifier(text, candidate_labels=candidate_labels)
print(result)

{'sequence': 'The new battery lasts 48 hours on a single charge.', 'labels': ['technology', 'sports', 'cooking'], 'scores': [0.9806022047996521, 0.013702603057026863, 0.005695158150047064]}


The output is a dict with:
- `sequence` — the original text
- `labels` — sorted from most to least likely
- `scores` — confidence for each label (sum ≈ 1.0 in single-label mode)

## Multi-label mode

With `multi_label=True` each label is scored independently (scores no longer sum to 1).
Use this when a text can belong to **more than one** category at the same time.

In [5]:
text_multilabel = "The athlete followed a strict diet and trained six hours a day to prepare for the Olympics."
labels_multilabel = ["sports", "nutrition", "politics", "technology"]

result_ml = classifier(text_multilabel, candidate_labels=labels_multilabel, multi_label=True)

for label, score in zip(result_ml["labels"], result_ml["scores"]):
    print(f"{label:15s} {score:.3f}")

sports          0.954
nutrition       0.930
technology      0.001
politics        0.000


## Batch processing — multiple texts at once

Pass a list of strings to classify several texts in one call.
This is faster than calling the pipeline in a Python loop because the model processes them together.

In [6]:
texts = [
    "Scientists discover a new species of deep-sea fish.",
    "The central bank raised interest rates by 0.5%.",
    "The team scored three goals in the last ten minutes.",
]
labels_batch = ["science", "economy", "sports", "politics"]

results = classifier(texts, candidate_labels=labels_batch)

for r in results:
    top_label = r["labels"][0]
    top_score = r["scores"][0]
    print(f"[{top_label} ({top_score:.2f})] {r['sequence'][:60]}...")

[science (0.98)] Scientists discover a new species of deep-sea fish....
[economy (0.74)] The central bank raised interest rates by 0.5%....
[sports (0.87)] The team scored three goals in the last ten minutes....


## Key takeaways

| Concept | What it means |
|---|---|
| Zero-shot | No task-specific training needed — works out of the box |
| NLI backbone | The model checks if the text *entails* each label description |
| `multi_label=False` | Labels compete (scores sum to ~1) — pick the best one |
| `multi_label=True` | Labels are independent — multiple can score high |
| Batch input | Pass a list of texts for more efficient inference |